# Projeto Fictus | Analise de Vendas — Bloco 1: Viabilidade Econômica


---

## Pergunta Central do Bloco
> **A empresa se financia ou consome capital à medida que cresce?**

---

## Contexto do Bloco

Antes de qualquer decisão de aquisição, é necessário entender se o negócio tem capacidade de se autofinanciar com crescimento. Um negócio que cresce com deterioração de margem é um passivo disfarçado de ativo.

Para avaliar a viabilidade econômica da empresa-alvo, a análise foi estruturada em hipóteses críticas de sustentabilidade, margem, escala e concentração, utilizando frameworks clássicos de gestão e engenharia de produção para organizar o raciocínio.

**Este bloco investiga:**
1. A receita apresenta crescimento consistente ou alta volatilidade?
2. O crescimento de receita é acompanhado por manutenção ou deterioração da margem operacional?
3. O custo de frete ao cliente representa uma barreira que compromete a sustentabilidade da demanda?
4. Há evidências de tração sustentável ou apenas resultados pontuais?
5. Em quais períodos o crescimento mascarou perda de rentabilidade?
6. Quais dimensões (produto, canal, região) mais explicam a variação de margem?
7. O crescimento do ticket médio acompanha o crescimento de volume — ou dilui com escala?
8. A receita cresceu por força própria do negócio ou foi impulsionada por contexto macroeconômico favorável?
9. A sazonalidade representa um padrão estrutural ou um risco de gestão de demanda?
10. Quantos clientes respondem por 80% da receita — há concentração de risco na base de compradores?
11. A base de clientes cresce por retenção ou por aquisição constante sem recompra — há churn silencioso?

---

## Nota Metodológica — Deslocamento Temporal
Os dados originais do dataset Olist compreendem o período **2016–2018**.  
Para fins de contextualização analítica, todas as datas foram deslocadas **+7 anos**, resultando no período **2023–2025**. Premissa declarada — intervalos preservados.  
**Análise restrita a partir de janeiro/2024** — os primeiros meses de operação (set–dez/2023) são desconsiderados por representarem rampa inicial, não operação madura.

---

## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

# ─── Caminhos relativos — funcionam em qualquer máquina ─────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_EXT      = BASE_DIR / "data" / "externos"
DIR_EXPORTS  = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)


warnings.filterwarnings("ignore")

# ─── Caminhos ─────────────────────────────────────────────────────────────────
# ─── Paleta e estilo ──────────────────────────────────────────────────────────
COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    elif abs(x) >= 1_000:   return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento dos Dados Internos

In [ ]:
# ─── Separador padrão: vírgula, decimal ponto ────────────────────────────────
#   Todos os arquivos gerados pelo novo ETL usam sep=',' e decimal='.'
def ler_csv(caminho, sep=",", **kwargs):
    df = pd.read_csv(caminho, sep=sep, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de datas ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega"]:
    fato[col] = pd.to_datetime(fato[col], dayfirst=False, errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

# ─── Colunas numéricas já vêm tipadas corretamente do ETL ────────────────────
# Garantia adicional caso haja leitura inesperada como object
COLS_NUMERICAS = ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
                  "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]
for col in COLS_NUMERICAS:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento ───────────────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto", how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente", how="left")
fato = fato.merge(dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]], on="id_data", how="left")

# ─── Filtro temporal: apenas dados a partir de jan/2024 ───────────────────────
DATA_INICIO = "2024-01-01"
fato = fato[fato["data_compra"] >= DATA_INICIO].copy()
fato_entregues = fato[fato["status_pedido"] == "entregue"].copy()

print(f"[FILTRO] Dados a partir de: {DATA_INICIO}")
print(f"fato (filtrado)   : {len(fato):>8} linhas | {fato['data_compra'].min().date()} → {fato['data_compra'].max().date()}")
print(f"Pedidos entregues : {len(fato_entregues):>8} linhas ({len(fato_entregues)/len(fato)*100:.1f}% do total)")


## Carregamento dos Dados Externos (Macro)

Cruzamento com IPCA (inflação) e taxa de desocupação PNADC do IBGE.  
Esses dados permitem separar crescimento real de crescimento inflacionário e verificar se a performance do negócio depende de contexto macroeconômico favorável.

> Se os arquivos não estiverem disponíveis, as análises macro são automaticamente ignoradas.

In [ ]:
def carregar_macro(nome_arquivo, nome_coluna_saida, filtro_ano_min=None):
    """Carrega série macro e retorna DataFrame padronizado com colunas: ano_mes e indicador."""
    caminho = DIR_EXT / nome_arquivo
    if not caminho.exists():
        print(f"  [AVISO] {nome_arquivo} não encontrado — análise macro parcialmente desativada")
        return None
    df = pd.read_csv(caminho, sep=";", encoding="latin-1")
    df.columns = df.columns.str.strip()
    df["ano_mes"] = pd.to_datetime(df["Periodo"], format="%m/%Y", errors="coerce") \
                     .dt.to_period("M").astype(str)
    df[nome_coluna_saida] = (
        df["Taxa"].astype(str).str.strip()
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )
    df = df[["ano_mes", nome_coluna_saida]].dropna().sort_values("ano_mes")
    if filtro_ano_min:
        df = df[df["ano_mes"] >= filtro_ano_min]
    print(f"  [OK] {nome_arquivo} — {len(df)} registros ({df['ano_mes'].min()} → {df['ano_mes'].max()})")
    return df

df_ipca  = carregar_macro("ipca_mensal.csv",       "ipca_pct",       filtro_ano_min="2024-01")
df_desoc = carregar_macro("desocupacao_pnadc.csv", "desocupacao_pct", filtro_ano_min="2024-01")

# ─── Série de receita mensal base ─────────────────────────────────────────────
receita_mensal = (
    fato_entregues
    .groupby("ano_mes")
    .agg(
        receita_bruta   = ("preco",               "sum"),
        receita_total   = ("valor_total_item",     "sum"),
        frete_total     = ("valor_frete",          "sum"),
        n_pedidos       = ("id_pedido",            "nunique"),
        nota_media      = ("nota_review",          "mean"),
    )
    .reset_index()
    .sort_values("ano_mes")
)
receita_mensal["pct_frete_receita"] = receita_mensal["frete_total"] / receita_mensal["receita_bruta"] * 100
receita_mensal["ticket_medio"]      = receita_mensal["receita_bruta"] / receita_mensal["n_pedidos"]
receita_mensal["crescimento_mom"]   = receita_mensal["receita_bruta"].pct_change(fill_method=None) * 100

if df_ipca  is not None: receita_mensal = receita_mensal.merge(df_ipca,  on="ano_mes", how="left")
if df_desoc is not None: receita_mensal = receita_mensal.merge(df_desoc, on="ano_mes", how="left")

print(f"\nreceita_mensal: {len(receita_mensal)} meses | {receita_mensal['ano_mes'].min()} → {receita_mensal['ano_mes'].max()}")
print(f"Receita total acumulada: R$ {receita_mensal['receita_bruta'].sum():,.0f}")

---

## Análise 1 — A receita apresenta crescimento consistente ou alta volatilidade?

> *"A análise da consistência da receita permite identificar se o crescimento observado é fruto de uma tendência sólida ou de eventos isolados e sazonais. Em processos de aquisição, a previsibilidade do faturamento é um fator determinante para a segurança do investimento, pois volatilidades extremas aumentam o risco de fluxo de caixa e dificultam o planejamento operacional de longo prazo."*

**Framework:** Ciclo PDCA — etapa Check  
**Entrega:** Série temporal de receita com média móvel 3 meses, crescimento MoM e volume de pedidos

**Como este script responde à pergunta:**
> Para medir consistência, não basta olhar o número final — é preciso ver o comportamento ao longo do tempo. Este script executa três leituras simultâneas:
>
> 1. **Série de receita com média móvel:** Plota a receita mês a mês e sobrepõe uma média móvel de 3 meses. A média móvel suaviza ruídos pontuais e revela a tendência real — se a linha laranja sobe consistentemente, o crescimento é estrutural; se oscila em torno da receita bruta, há volatilidade relevante.
> 2. **Crescimento mês a mês (MoM):** Cada barra mostra quanto a receita variou em relação ao mês anterior, em percentual. Barras verdes indicam aceleração; barras vermelhas indicam queda. Um negócio saudável tem predominância de verde com quedas pequenas e isoladas — não sequências de vermelho.
> 3. **Volume de pedidos:** Mostra se o crescimento de receita vem de mais pedidos ou de ticket maior. Se receita sobe mas pedidos caem, pode haver concentração em poucos clientes de alto valor — um risco oculto.

**Análise do Resultado:** 
Este indicador serve para medir a previsibilidade do fluxo de caixa. Um crescimento consistente e suave indica um negócio maduro e com domínio de mercado, facilitando o planejamento de longo prazo. Já uma volatilidade alta (muitos altos e baixos) sugere um negócio dependente de eventos externos ou promoções agressivas, o que aumenta o risco para um investidor, pois o faturamento de amanhã é sempre uma incerteza.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle("Análise 1 — Série Temporal de Receita", fontsize=14, fontweight="bold", y=1.01)

x      = range(len(receita_mensal))
labels = receita_mensal["ano_mes"].tolist()

# Subplot 1: Receita + média móvel 3m
axes[0].bar(x, receita_mensal["receita_bruta"] / 1000, color=COR_RECEITA, alpha=0.7, label="Receita mensal")
mm3 = receita_mensal["receita_bruta"].rolling(3).mean() / 1000
axes[0].plot(x, mm3, color=COR_DESTAQUE, linewidth=2, label="Média móvel 3m", zorder=5)
axes[0].set_ylabel("Receita (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
axes[0].legend(frameon=False)
axes[0].set_title("Receita Bruta Mensal (preço dos produtos)", fontsize=11, pad=8)

# Subplot 2: Crescimento MoM
cores_mom = [COR_MARGEM if v >= 0 else COR_ALERTA for v in receita_mensal["crescimento_mom"].fillna(0)]
axes[1].bar(x, receita_mensal["crescimento_mom"].fillna(0), color=cores_mom, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_ylabel("Variação MoM (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Crescimento Mês a Mês (MoM)", fontsize=11, pad=8)

# Subplot 3: Volume de pedidos
axes[2].plot(x, receita_mensal["n_pedidos"], color=COR_ROXO, linewidth=2, marker="o", markersize=3)
axes[2].fill_between(x, receita_mensal["n_pedidos"], alpha=0.15, color=COR_ROXO)
axes[2].set_ylabel("Nº de Pedidos")
axes[2].set_title("Volume de Pedidos Entregues por Mês", fontsize=11, pad=8)

tick_idx = list(range(0, len(labels), 3))
axes[2].set_xticks(tick_idx)
axes[2].set_xticklabels([labels[i] for i in tick_idx], rotation=45, ha="right", fontsize=8)

plt.tight_layout()
salvar(fig, "01_serie_receita")
plt.show()

# Insight automático
receita_max = receita_mensal.loc[receita_mensal["receita_bruta"].idxmax()]
receita_min = receita_mensal.loc[receita_mensal["receita_bruta"].idxmin()]
rm_v = receita_mensal.dropna(subset=["crescimento_mom"])
meses_pos = (rm_v["crescimento_mom"] > 0).sum()
meses_neg = (rm_v["crescimento_mom"] < 0).sum()

print("\n" + "="*55)
print("INSIGHT — SÉRIE TEMPORAL DE RECEITA")
print("="*55)
print(f"Mês de pico     : {receita_max['ano_mes']} — R$ {receita_max['receita_bruta']:,.0f}")
print(f"Mês de vale     : {receita_min['ano_mes']} — R$ {receita_min['receita_bruta']:,.0f}")
print(f"Razão pico/vale : {receita_max['receita_bruta']/receita_min['receita_bruta']:.1f}x")
print(f"Meses positivos : {meses_pos} de {len(rm_v)} ({meses_pos/len(rm_v)*100:.0f}%)")
print(f"Meses negativos : {meses_neg} de {len(rm_v)} ({meses_neg/len(rm_v)*100:.0f}%)")

---

## Análise 2 — O crescimento de receita é acompanhado por manutenção ou deterioração da margem operacional?

> *"É fundamental verificar se a escala está gerando eficiência ou se a operação está se tornando mais cara proporcionalmente ao que vende. O monitoramento da margem operacional em conjunto com o faturamento revela a "saúde" do crescimento: Um crescimento de receita acompanhado de aumento proporcional do frete indica que a operação está se tornando menos eficiente por real gerado — sinal de alerta relevante para a decisão de aquisição."*

**Framework:** Controle de processo   
**Entrega:** Decomposição de margem por trimestre com identificação de períodos de deterioração

**Como este script responde à pergunta:**
> Crescimento de receita sem controle de rentabilidade é uma armadilha clássica. Este script decompõe a receita em dois componentes e os analisa juntos:
>
> 1. **Composição por trimestre:** Empilha receita de produto e frete pago pelo cliente. Isso revela se o frete está ganhando peso proporcional na receita total — o que indica pressão crescente sobre o consumidor.
> 2. **% Frete sobre receita:** Isola o percentual que o frete representa da receita bruta e compara com a mediana do período. Trimestres com barras vermelhas estão acima da mediana — sinal de que a eficiência logística piorou ou o mix de produtos mudou para categorias com frete mais pesado.
>
> A leitura conjunta responde a pergunta: se a receita cresce mas o % de frete também sobe, o crescimento está vindo com deterioração silenciosa de margem.

**Análise do Resultado:**
 É o teste real da saúde do negócio. Se a receita sobe, mas a margem cai, a empresa está "trabalhando mais para ganhar menos". O ideal é observar a manutenção ou ganho de margem com o volume, o que prova que a operação é eficiente e possui o que chamamos de alavancagem operacional.


In [ ]:
margem_trimestral = (
    fato_entregues
    .groupby(["ano", "trimestre"])
    .agg(
        receita_bruta = ("preco",       "sum"),
        frete_total   = ("valor_frete", "sum"),
        n_pedidos     = ("id_pedido",   "nunique"),
    )
    .reset_index()
)
margem_trimestral["label"]             = margem_trimestral["ano"].astype(str) + "-Q" + margem_trimestral["trimestre"].astype(str)
margem_trimestral["pct_frete_receita"] = margem_trimestral["frete_total"] / margem_trimestral["receita_bruta"] * 100
margem_trimestral["receita_por_pedido"]= margem_trimestral["receita_bruta"] / margem_trimestral["n_pedidos"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 2 — Margem Operacional Proxy por Trimestre", fontsize=13, fontweight="bold")

x      = range(len(margem_trimestral))
labels = margem_trimestral["label"].tolist()

# Receita e frete empilhados
axes[0].bar(x, margem_trimestral["receita_bruta"] / 1000, label="Preço produto", color=COR_RECEITA, alpha=0.85)
axes[0].bar(x, margem_trimestral["frete_total"] / 1000,
            bottom=margem_trimestral["receita_bruta"] / 1000,
            label="Frete (pago pelo cliente)", color="#85C1E9", alpha=0.85)
axes[0].set_title("Composição de Receita por Trimestre", fontsize=11)
axes[0].set_ylabel("Receita (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
axes[0].legend(frameon=False)

# % Frete
mediana_frete = margem_trimestral["pct_frete_receita"].median()
cores_frete = [COR_ALERTA if v > mediana_frete + 3 else COR_NEUTRO for v in margem_trimestral["pct_frete_receita"]]
axes[1].bar(x, margem_trimestral["pct_frete_receita"], color=cores_frete, alpha=0.85)
axes[1].axhline(mediana_frete, color=COR_DESTAQUE, linestyle="--", linewidth=1.5,
                label=f"Mediana: {mediana_frete:.1f}%")
axes[1].set_title("% Frete sobre Receita — Barreira de Conversão", fontsize=11)
axes[1].set_ylabel("Frete / Receita (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
axes[1].legend(frameon=False)

plt.tight_layout()
salvar(fig, "02_margem_proxy")
plt.show()

print("\n" + "="*55)
print("INSIGHT — MARGEM OPERACIONAL PROXY")
print("="*55)
print(f"% Frete médio    : {margem_trimestral['pct_frete_receita'].mean():.1f}%")
print(f"% Frete mínimo   : {margem_trimestral['pct_frete_receita'].min():.1f}% ({margem_trimestral.loc[margem_trimestral['pct_frete_receita'].idxmin(), 'label']})")
print(f"% Frete máximo   : {margem_trimestral['pct_frete_receita'].max():.1f}% ({margem_trimestral.loc[margem_trimestral['pct_frete_receita'].idxmax(), 'label']})")
frete_trend = np.polyfit(list(x), margem_trimestral["pct_frete_receita"], 1)[0]
tend = "↑ crescendo (pressão de demanda aumentando)" if frete_trend > 0.1 else "↓ caindo (positivo)" if frete_trend < -0.1 else "→ estável"
print(f"Tendência        : {tend}")

---

## Análise 3 — O custo de frete ao cliente representa uma barreira que compromete a sustentabilidade da demanda?

> *"O frete atua como um regulador direto da conversão de vendas e da satisfação do cliente. Investigar o peso do transporte sobre o ticket médio ajuda a entender a sensibilidade do modelo de negócio a choques logísticos. Se o custo logístico cresce acima da capacidade de absorção do consumidor, a sustentabilidade da demanda futura fica comprometida — risco que precisa ser quantificado antes de qualquer decisão de aquisição."*

**Framework:** Ciclo PDCA — análise de coerência entre crescimento de volume e sustentabilidade da demanda  
**Entrega:** Evolução do % frete sobre receita como indicador de barreira de conversão, com correlação com nota de review e volume de pedidos

**Como este script responde à pergunta:**
> O frete é invisível na demonstração de resultados da operação — mas é muito visível para o consumidor. Este script aplica três testes para medir seu impacto real na demanda:
>
> 1. **Histórico de peso:** Rastreia a evolução do % de frete sobre receita mês a mês e adiciona uma linha de tendência. Se a tendência sobe, o frete está pesando cada vez mais — mesmo que a receita nominal cresça.
> 2. **Teste de satisfação:** Calcula a correlação estatística entre o % de frete e a nota média de review dos clientes. Uma correlação negativa significativa (abaixo de -0,3 com p < 0,05) confirma que frete alto prejudica a experiência percebida.
> 3. **Barreira de volume:** Correlaciona o % de frete com o número de pedidos realizados no mês. Correlação negativa aqui significa que nos meses em que o frete pesou mais, menos pedidos foram feitos — evidência de abandono de carrinho ou decisão de não comprar.
>
> Os três resultados juntos formam um diagnóstico: se as correlações forem negativas e significativas, o frete já é uma barreira ativa de conversão, não apenas um custo de conveniência.

**Análise do Resultado:**
 O frete é, muitas vezes, o "vilão silencioso" do e-commerce. Se o custo logístico cresce a ponto de se aproximar do valor do produto, a empresa perde competitividade. Identificamos aqui se o cliente está desistindo da compra pelo preço da entrega.


In [ ]:
# Correlação entre % frete e nota de review / volume
frete_nota = receita_mensal[["pct_frete_receita", "nota_media", "n_pedidos"]].dropna()
r_frete_nota, p_frete_nota = stats.pearsonr(frete_nota["pct_frete_receita"], frete_nota["nota_media"])
r_frete_vol,  p_frete_vol  = stats.pearsonr(frete_nota["pct_frete_receita"], frete_nota["n_pedidos"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Análise 3 — Frete como Barreira de Demanda", fontsize=13, fontweight="bold")

x      = range(len(receita_mensal))
xtick  = list(range(0, len(receita_mensal), 3))
xlabs  = [receita_mensal["ano_mes"].iloc[i] for i in xtick]

# Evolução temporal do % frete
axes[0].plot(x, receita_mensal["pct_frete_receita"], color=COR_ALERTA, linewidth=2, marker="o", markersize=3)
z = np.polyfit(list(x), receita_mensal["pct_frete_receita"].fillna(method="ffill"), 1)
axes[0].plot(x, np.poly1d(z)(list(x)), color="black", linewidth=1, linestyle=":", alpha=0.5)
axes[0].axhline(receita_mensal["pct_frete_receita"].mean(), color=COR_NEUTRO, linestyle="--",
                linewidth=1, label=f"Média: {receita_mensal['pct_frete_receita'].mean():.1f}%")
axes[0].set_title("% Frete sobre Receita ao Longo do Tempo", fontsize=11)
axes[0].set_ylabel("Frete / Receita (%)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_xticks(xtick); axes[0].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
axes[0].legend(frameon=False, fontsize=8)

# Frete × Nota de review
axes[1].scatter(receita_mensal["pct_frete_receita"], receita_mensal["nota_media"],
                color=COR_ALERTA, alpha=0.7, s=60, edgecolors="white", linewidth=0.5)
if len(frete_nota) > 3:
    m, b, _, _, _ = stats.linregress(frete_nota["pct_frete_receita"], frete_nota["nota_media"])
    xfit = np.linspace(frete_nota["pct_frete_receita"].min(), frete_nota["pct_frete_receita"].max(), 50)
    axes[1].plot(xfit, m*xfit+b, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
axes[1].set_xlabel("% Frete sobre Receita")
axes[1].set_ylabel("Nota Média de Review")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title(f"Frete × Satisfação do Cliente\ncorr={r_frete_nota:.2f} (p={p_frete_nota:.3f})", fontsize=11)

# Frete × Volume
axes[2].scatter(receita_mensal["pct_frete_receita"], receita_mensal["n_pedidos"],
                color=COR_RECEITA, alpha=0.7, s=60, edgecolors="white", linewidth=0.5)
if len(frete_nota) > 3:
    m2, b2, _, _, _ = stats.linregress(frete_nota["pct_frete_receita"], frete_nota["n_pedidos"])
    axes[2].plot(xfit, m2*xfit+b2, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
axes[2].set_xlabel("% Frete sobre Receita")
axes[2].set_ylabel("Nº de Pedidos no Mês")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[2].set_title(f"Frete × Volume de Pedidos\ncorr={r_frete_vol:.2f} (p={p_frete_vol:.3f})", fontsize=11)

plt.tight_layout()
salvar(fig, "03_frete_barreira_demanda")
plt.show()

print("\n" + "="*55)
print("INSIGHT — FRETE COMO BARREIRA DE DEMANDA")
print("="*55)
print(f"% Frete médio sobre receita   : {receita_mensal['pct_frete_receita'].mean():.1f}%")
print(f"Corr. frete × nota de review  : {r_frete_nota:.2f} — {'frete alto reduz satisfação' if r_frete_nota < -0.2 else 'sem relação clara'}")
print(f"Corr. frete × volume          : {r_frete_vol:.2f} — {'frete alto reduz volume' if r_frete_vol < -0.2 else 'sem relação clara'}")
print(f"\nConexão com Fase 2 — Logística (Fase 2):")
print(f"  Pergunta central: vale internalizar logística para reduzir frete ao cliente?")

---

## Análise 4 — Há evidências de tração sustentável ou apenas resultados pontuais?

> *"A diferenciação entre tração real e resultados acidentais é feita pela análise do padrão de crescimento ao longo do tempo. Resultados pontuais podem mascarar fragilidades estruturais. Validar a consistência da tendência garante que a tese de investimento esteja ancorada em um comportamento replicável, não em anomalias de mercado ou eventos sazonais isolados."*

**Framework:** PDCA — análise de tendência  
**Entrega:** Gráfico de break-even mensal com classificação de meses positivos e negativos

**Como este script responde à pergunta:**
> Um resultado positivo isolado pode ser coincidência. Este script distingue entre resultado pontual e padrão sustentado de duas formas:
>
> 1. **Deflação pelo IPCA (quando disponível):** Converte a receita nominal em receita real, removendo o efeito da inflação. Isso é crítico: um negócio que cresce 8% ao ano com IPCA em 6% está crescendo apenas 2% em termos reais. Se os dados macro estiverem disponíveis, o gráfico mostra o crescimento real; caso contrário, usa o nominal e avisa.
> 2. **Classificação mês a mês:** Cada barra é colorida de verde (crescimento positivo) ou vermelho (queda). Os meses negativos recebem anotação com o percentual exato de queda — facilitando a identificação de padrões como quedas sazonais ou sequências de deterioração.
>
> O insight automático conta quantos meses foram positivos e negativos, e identifica o pior e o melhor mês. Se mais de 70% dos meses forem positivos sem sequências longas de queda, o break-even é sustentável.

**Análise do Resultado:**
 Buscamos diferenciar "sorte" de "estratégia". Resultados pontuais podem ser causados por uma única grande campanha ou um mês atípico. A tração sustentável é demonstrada quando os indicadores mantêm uma tendência de alta por vários períodos consecutivos, provando que o mercado realmente deseja o produto da empresa-alvo de forma contínua.


In [ ]:
rm = receita_mensal.copy()

if "ipca_pct" in rm.columns and rm["ipca_pct"].notna().sum() > 10:
    rm["ipca_fator"]       = (1 + rm["ipca_pct"] / 100).cumprod()
    rm["receita_real"]     = rm["receita_bruta"] / rm["ipca_fator"]
    rm["crescimento_real"] = rm["receita_real"].pct_change(fill_method=None) * 100
    usar_real = True
    print("[OK] Usando receita deflacionada pelo IPCA")
else:
    # Sem IPCA: usa crescimento acumulado indexado (base 100 no mês inicial)
    rm["receita_idx"] = rm["receita_bruta"] / rm["receita_bruta"].iloc[0] * 100
    rm["crescimento_real"] = rm["receita_idx"].pct_change(fill_method=None) * 100
    usar_real = False
    print("[INFO] IPCA não disponível — usando índice de crescimento acumulado (base 100)")

rm["positivo"] = rm["crescimento_real"] > 0

fig, ax = plt.subplots(figsize=(14, 5))
titulo = "(Real, deflacionado pelo IPCA)" if usar_real else "(Nominal)"
ax.set_title(f"Análise 4 — Tração de Crescimento {titulo}", fontsize=12, fontweight="bold")

x = range(len(rm))
cores_be = [COR_MARGEM if v else COR_ALERTA for v in rm["positivo"]]
ax.bar(x, rm["crescimento_real"].fillna(0), color=cores_be, alpha=0.85)
ax.axhline(0, color="black", linewidth=1)

negativos = rm[rm["crescimento_real"] < 0]
for idx, row in negativos.iterrows():
    pos = rm.index.get_loc(idx)
    ax.annotate(f"{row['crescimento_real']:.1f}%",
                xy=(pos, row["crescimento_real"]),
                xytext=(pos, row["crescimento_real"] - 2),
                fontsize=7, ha="center", color=COR_ALERTA)

tick_idx = list(range(0, len(rm), 3))
ax.set_xticks(tick_idx)
ax.set_xticklabels([rm["ano_mes"].iloc[i] for i in tick_idx], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Crescimento MoM (%)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax.legend(handles=[
    mpatches.Patch(color=COR_MARGEM, label="Crescimento positivo"),
    mpatches.Patch(color=COR_ALERTA, label="Queda"),
], frameon=False)

plt.tight_layout()
salvar(fig, "04_break_even_mensal")
plt.show()

n_pos   = rm["positivo"].sum()
n_total = rm["positivo"].notna().sum()
print("\n" + "="*55)
print("INSIGHT — TRAÇÃO MENSAL")
print("="*55)
print(f"Meses analisados  : {n_total}")
print(f"Meses positivos   : {n_pos} ({n_pos/n_total*100:.1f}%)")
print(f"Meses negativos   : {n_total-n_pos} ({(n_total-n_pos)/n_total*100:.1f}%)")
print(f"Maior queda MoM   : {rm['crescimento_real'].min():.1f}% ({rm.loc[rm['crescimento_real'].idxmin(), 'ano_mes']})")
print(f"Maior alta MoM    : {rm['crescimento_real'].max():.1f}% ({rm.loc[rm['crescimento_real'].idxmax(), 'ano_mes']})")

---

## Análise 5 — Em quais períodos o crescimento mascarou perda de rentabilidade?

> *"IIdentificar períodos históricos onde o volume de vendas subiu enquanto o % de frete sobre receita também cresceu é essencial para detectar ineficiências ocultas. Essa leitura temporal permite correlacionar picos de demanda com deterioração silenciosa de eficiência — revelando se o crescimento passado foi estruturalmente saudável ou construído sobre bases frágeis."*

**Framework:** Controle de processo   
**Entrega:** Quadrante crescimento × % frete por trimestre, identificando períodos de crescimento com deterioração simultânea

**Como este script responde à pergunta:**
> Este script aplica um quadrante de quatro zonas para classificar cada trimestre segundo dois eixos simultâneos — crescimento de receita e variação do % de frete:
>
> 1. **Crescimento saudável (verde):** Receita subiu e % frete caiu ou ficou estável. O negócio cresceu sem comprometer eficiência.
> 2. **Crescimento mascarando pressão (laranja):** Receita subiu, mas % frete também subiu. O volume cresce, mas a estrutura está ficando mais cara por real gerado — risco que se acumula silenciosamente.
> 3. **Retração com melhora de eficiência (cinza):** Receita caiu, mas % frete melhorou. Pode ser ajuste estratégico de mix.
> 4. **Deterioração total (vermelho):** Receita caiu e % frete piorou. O pior cenário possível.
>
> Cada trimestre aparece plotado e identificado pelo seu rótulo. O padrão de distribuição revela se a empresa tem disciplina operacional ou se o crescimento histórico foi construído sobre fundamentos frágeis.

**Análise do Resultado:**
 Muitas vezes, um faturamento recorde esconde um prejuízo operacional. Esta análise identifica momentos onde a empresa acelerou as vendas, mas perdeu a mão nos custos. Para um comprador de M&A, detectar esses períodos é vital para não comprar um "voo de galinha" — um negócio que parece grande, mas que não se sustenta financeiramente.

In [ ]:
# Calcula crescimento de receita e variação de % frete por trimestre
mt = margem_trimestral.copy().sort_values("label")
mt["cresc_receita"]  = mt["receita_bruta"].pct_change(fill_method=None) * 100
mt["delta_frete_pp"] = mt["pct_frete_receita"].diff()
mt = mt.dropna(subset=["cresc_receita", "delta_frete_pp"])

def classificar(row):
    if   row["cresc_receita"] >= 0 and row["delta_frete_pp"] <= 0: return "Crescimento saudável"
    elif row["cresc_receita"] >= 0 and row["delta_frete_pp"] >  0: return "Crescimento mascarando pressão"
    elif row["cresc_receita"] <  0 and row["delta_frete_pp"] <= 0: return "Retração com melhora de eficiência"
    else: return "Deterioração total"

mt["quadrante"] = mt.apply(classificar, axis=1)
paleta_q = {
    "Crescimento saudável":            COR_MARGEM,
    "Crescimento mascarando pressão":  COR_DESTAQUE,
    "Retração com melhora de eficiência": COR_NEUTRO,
    "Deterioração total":              COR_ALERTA,
}

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_title("Análise 5 — Quadrante: Crescimento × Pressão de Frete por Trimestre",
             fontsize=12, fontweight="bold")

for q, grupo in mt.groupby("quadrante"):
    ax.scatter(grupo["cresc_receita"], grupo["delta_frete_pp"],
               s=150, color=paleta_q[q], alpha=0.85, label=q,
               edgecolors="white", linewidth=0.5, zorder=5)
    for _, row in grupo.iterrows():
        ax.annotate(row["label"], (row["cresc_receita"], row["delta_frete_pp"]),
                    fontsize=8, xytext=(5, 5), textcoords="offset points")

ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Crescimento de Receita vs Trimestre Anterior (%)")
ax.set_ylabel("Variação do % Frete sobre Receita (pp)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.1f}pp"))

# Rótulos dos quadrantes
xlim, ylim = ax.get_xlim(), ax.get_ylim()
ax.text(xlim[1]*0.6, ylim[1]*0.85, "Crescimento saudável",         color=COR_MARGEM,   fontsize=9, alpha=0.6)
ax.text(xlim[1]*0.6, ylim[0]*0.85, "Crescimento mascarando pressão", color=COR_DESTAQUE, fontsize=9, alpha=0.6)
ax.text(xlim[0]*0.9, ylim[0]*0.85, "Deterioração total",           color=COR_ALERTA,   fontsize=9, alpha=0.6)
ax.legend(frameon=False, fontsize=8, loc="upper left")

plt.tight_layout()
salvar(fig, "05_quadrante_crescimento_margem")
plt.show()

print("\n" + "="*55)
print("INSIGHT — QUADRANTE CRESCIMENTO × MARGEM")
print("="*55)
for q, grupo in mt.groupby("quadrante"):
    trimestres = ", ".join(grupo["label"].tolist())
    print(f"  {q}: {trimestres}")

---

## Análise 6 — Quais dimensões (produto, canal, região) mais explicam a variação de margem?

> *"A aplicação do Princípio de Pareto permite decompor a receita para identificar onde o faturamento está de fato concentrado, saindo da visão limitada das médias agregadas. Identificar se poucas categorias ou estados explicam a maior parte do resultado é essencial para mapear o risco de concentração. Qualquer mudança logística ou competitiva nesse núcleo tem impacto imediato e desproporcional no resultado consolidado — informação que o comprador precisa ter antes de fechar o negócio."*

**Framework:** Pareto (80/20)  
**Entrega:** Decomposição de margem por categoria e por estado, com curva de Pareto

**Como este script responde à pergunta:**
> Médias agregadas escondem mais do que revelam. Este script aplica Pareto em duas dimensões para identificar onde a margem realmente está concentrada — e onde os riscos estão ocultos:
>
> 1. **Pareto por categoria:** Lista as 15 categorias com maior receita em ordem crescente (para facilitar leitura horizontal) e anota ao lado de cada barra o percentual de receita que representa e o % de frete associado. Categorias com frete alto e receita relevante são candidatas a revisão de precificação ou logística.
> 2. **Curva de Pareto por estado:** Ordena os estados por receita e plota a curva acumulada. A linha pontilhada vertical marca onde 80% da receita é atingida. O resultado diz quantos estados concentram a maior parte do faturamento — e quais estados fora desse grupo têm custo de frete desproporcional.
>
> A leitura combinada das duas dimensões indica onde focar esforços: se poucas categorias em poucos estados explicam 80% da receita, qualquer mudança nesse núcleo tem impacto imediato no resultado.

**Análise do Resultado:**
 O objetivo aqui é "abrir a caixa preta" do resultado operacional estimado. Descobrimos se o dinheiro está vindo de uma região específica (ex: Sudeste) ou de um grupo seleto de produtos. Isso revela onde o negócio é forte e onde ele está apenas "gastando energia". Se o lucro depende de uma única dimensão, o risco de concentração é alto.


In [ ]:
# Margem por categoria
margem_cat = (
    fato_entregues
    .groupby("nome_categoria_produto")
    .agg(receita_bruta=("preco", "sum"), frete_total=("valor_frete", "sum"), n_itens=("id_pedido", "count"))
    .reset_index()
)
margem_cat["pct_receita"] = margem_cat["receita_bruta"] / margem_cat["receita_bruta"].sum() * 100
margem_cat["pct_frete"]   = margem_cat["frete_total"]   / margem_cat["receita_bruta"] * 100
margem_cat["pct_acum"]    = margem_cat.sort_values("receita_bruta", ascending=False)["pct_receita"].cumsum().values

# Margem por estado
margem_estado = (
    fato_entregues
    .groupby("estado_cliente")
    .agg(receita_bruta=("preco", "sum"), frete_total=("valor_frete", "sum"), n_pedidos=("id_pedido", "nunique"))
    .reset_index()
    .sort_values("receita_bruta", ascending=False)
)
margem_estado["pct_receita"] = margem_estado["receita_bruta"] / margem_estado["receita_bruta"].sum() * 100
margem_estado["pct_frete"]   = margem_estado["frete_total"]   / margem_estado["receita_bruta"] * 100
margem_estado["pct_acum"]    = margem_estado["pct_receita"].cumsum()

top15 = margem_cat.sort_values("receita_bruta", ascending=False).head(15).sort_values("receita_bruta", ascending=True)
piores_frete = margem_cat.nlargest(3, "pct_frete")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 6 — Decomposição de Margem: Pareto por Categoria e Estado", fontsize=13, fontweight="bold")

# Categoria: receita + % frete
ax1 = axes[0]
bars = ax1.barh([c.replace("_", " ")[:30] for c in top15["nome_categoria_produto"]],
                top15["receita_bruta"] / 1000, color=COR_RECEITA, alpha=0.85)
ax1.set_xlabel("Receita (R$ mil)")
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
ax1.set_title("Receita por Categoria (Top 15)", fontsize=11)
for bar, pct_r, pct_f in zip(bars, top15["pct_receita"], top15["pct_frete"]):
    ax1.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f"{pct_r:.1f}% | frete: {pct_f:.1f}%", va="center", fontsize=7, color=COR_NEUTRO)

# Estado: curva acumulada de Pareto
ax2 = axes[1]
n_estados = len(margem_estado)
idx_80    = (margem_estado["pct_acum"] >= 80).idxmax()
pos_80    = margem_estado.index.get_loc(idx_80) + 1
ax2.plot(range(n_estados), margem_estado["pct_acum"].values, color=COR_RECEITA, linewidth=2, marker="o", markersize=4)
ax2.axhline(80, color=COR_ALERTA, linestyle="--", linewidth=1)
ax2.axvline(pos_80 - 1, color=COR_DESTAQUE, linestyle=":", linewidth=1.5)
ax2.annotate(f"{pos_80} estados = 80% da receita",
             xy=(pos_80 - 1, 80), xytext=(pos_80, 70), fontsize=9, color=COR_DESTAQUE,
             arrowprops=dict(arrowstyle="->", color=COR_DESTAQUE, lw=1))
ax2.set_title("Concentração de Receita por Estado (Pareto)", fontsize=11)
ax2.set_xlabel("Estados (ordenados por receita)")
ax2.set_ylabel("% Receita Acumulada")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax2.set_xticks(range(n_estados))
ax2.set_xticklabels(margem_estado["estado_cliente"].tolist(), rotation=45, ha="right", fontsize=7)

plt.tight_layout()
salvar(fig, "06_pareto_categoria_estado")
plt.show()

n_cats_80 = (margem_cat.sort_values("receita_bruta", ascending=False)["pct_receita"].cumsum() <= 80).sum() + 1
print("\n" + "="*55)
print("INSIGHT — PARETO DE CATEGORIAS E ESTADOS")
print("="*55)
print(f"Categorias para 80% da receita : {n_cats_80} de {len(margem_cat)}")
print(f"Estados para 80% da receita    : {pos_80} de {n_estados}")
print(f"\nCategorias com maior pressão de frete:")
for _, r in piores_frete.iterrows():
    print(f"  {r['nome_categoria_produto']:<38} frete: {r['pct_frete']:.1f}% | receita: {r['pct_receita']:.1f}%")

---

## Análise 7 — O crescimento do ticket médio acompanha o crescimento de volume — ou dilui com escala?

> *"O ticket médio atua como um indicador de saúde da marca e da qualidade da receita. Um crescimento de faturamento impulsionado apenas pelo volume, com queda no valor por transação, sugere uma "comoditização" do negócio e possível erosão de margem. Validar se a escala agrega ou dilui valor por pedido permite diferenciar um crescimento saudável de uma expansão baseada em produtos de baixo valor, que costumam ser mais sensíveis a custos logísticos."*

**Framework:** Controle de processo   
**Entrega:** Evolução do ticket médio mensal com análise de tendência e correlação com volume de pedidos

**Como este script responde à pergunta:**
> Ticket médio é um indicador de saúde silencioso — ele revela se o crescimento está vindo de mais valor por transação ou apenas de mais volume de pedidos menores. Este script faz duas leituras:
>
> 1. **Evolução temporal com tendência:** Plota o ticket médio mês a mês e adiciona uma linha de tendência. Se a tendência for descendente enquanto o volume de pedidos sobe, o crescimento está sendo puxado por pedidos de menor valor — um sinal clássico de diluição. A área sombreada em relação à média facilita a visualização dos períodos acima e abaixo do padrão histórico.
> 2. **Correlação volume × ticket:** O scatter mostra cada mês como um ponto, com o número de pedidos no eixo horizontal e o ticket no eixo vertical. A linha de regressão e o coeficiente de correlação revelam a relação: correlação negativa significa que mais pedidos vêm com tickets menores — a escala dilui o valor por transação em vez de sustentá-lo.

**Análise do Resultado:**
 Em um cenário ideal, a empresa vende mais e para clientes que gastam mais (ticket médio subindo). Se o volume aumenta, mas o ticket médio cai bruscamente, a empresa pode estar se tornando uma "comodidade", dependendo apenas de vender itens baratos e em massa para sobreviver, o que diminui o valor da marca no longo prazo.

In [ ]:
r_ticket_vol, p_ticket_vol = stats.pearsonr(
    receita_mensal["ticket_medio"].dropna(),
    receita_mensal["n_pedidos"].dropna()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 7 — Ticket Médio: Crescimento Agrega ou Dilui Valor?", fontsize=13, fontweight="bold")

x     = range(len(receita_mensal))
xtick = list(range(0, len(receita_mensal), 3))
xlabs = [receita_mensal["ano_mes"].iloc[i] for i in xtick]

# Evolução do ticket médio
ax1 = axes[0]
ax1.plot(x, receita_mensal["ticket_medio"], color=COR_RECEITA, linewidth=2, marker="o", markersize=3)
ax1.fill_between(x, receita_mensal["ticket_medio"], receita_mensal["ticket_medio"].mean(), alpha=0.1, color=COR_RECEITA)
ax1.axhline(receita_mensal["ticket_medio"].mean(), color=COR_NEUTRO, linestyle="--", linewidth=1,
            label=f"Média: R$ {receita_mensal['ticket_medio'].mean():.2f}")
# Linha de tendência
z = np.polyfit(list(x), receita_mensal["ticket_medio"].fillna(method="ffill"), 1)
ax1.plot(x, np.poly1d(z)(list(x)), color="black", linewidth=1, linestyle=":", alpha=0.5)
tend = "↑ crescendo (positivo)" if z[0] > 0.5 else "↓ caindo (diluição de valor)" if z[0] < -0.5 else "→ estável"
ax1.text(0.03, 0.93, f"Tendência: {tend}", transform=ax1.transAxes,
         fontsize=9, color=COR_MARGEM if z[0] > 0 else COR_ALERTA)
ax1.set_title("Evolução do Ticket Médio (R$ por pedido)", fontsize=11)
ax1.set_ylabel("R$ por pedido")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}"))
ax1.set_xticks(xtick); ax1.set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)
ax1.legend(frameon=False, fontsize=8)

# Scatter ticket × volume
ax2 = axes[1]
ax2.scatter(receita_mensal["n_pedidos"], receita_mensal["ticket_medio"],
            color=COR_RECEITA, alpha=0.7, s=60, edgecolors="white", linewidth=0.5)
tm = receita_mensal[["n_pedidos", "ticket_medio"]].dropna()
if len(tm) > 3:
    m, b, _, _, _ = stats.linregress(tm["n_pedidos"], tm["ticket_medio"])
    xfit = np.linspace(tm["n_pedidos"].min(), tm["n_pedidos"].max(), 50)
    ax2.plot(xfit, m*xfit+b, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
ax2.set_xlabel("Nº de Pedidos no Mês")
ax2.set_ylabel("Ticket Médio (R$)")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}"))
interp = "volume maior = ticket menor (diluição)" if r_ticket_vol < -0.3 else \
         "volume maior = ticket maior (escala saudável)" if r_ticket_vol > 0.3 else "sem relação clara"
ax2.set_title(f"Volume × Ticket Médio\ncorr={r_ticket_vol:.2f} — {interp}", fontsize=11)

plt.tight_layout()
salvar(fig, "07_ticket_medio")
plt.show()

print("\n" + "="*55)
print("INSIGHT — TICKET MÉDIO")
print("="*55)
print(f"Ticket médio global          : R$ {fato_entregues['preco'].mean():,.2f}")
print(f"Ticket máximo (mês)          : R$ {receita_mensal['ticket_medio'].max():,.2f} ({receita_mensal.loc[receita_mensal['ticket_medio'].idxmax(), 'ano_mes']})")
print(f"Ticket mínimo (mês)          : R$ {receita_mensal['ticket_medio'].min():,.2f} ({receita_mensal.loc[receita_mensal['ticket_medio'].idxmin(), 'ano_mes']})")
print(f"Correlação volume × ticket   : {r_ticket_vol:.2f} — {interp}")

---

## Análise 8 — A receita cresceu por força própria ou foi impulsionada por contexto macroeconômico favorável?

> *"Esta análise separa o mérito operacional do impacto de variáveis externas, como inflação (IPCA) e taxas de desocupação. O objetivo é validar se o crescimento é estrutural — baseado em vantagem competitiva e ganho de mercado — ou circunstancial, movido por um cenário econômico favorável. Negócios com alta dependência macroeconômica apresentam maior risco em cenários de recessão, o que deve ser ponderado no cálculo do prêmio de risco da aquisição."*

**Framework:** PDCA — análise de contexto e controle externo  
**Entrega:** Correlação entre receita mensal e IPCA / taxa de desocupação PNADC, com interpretação do coeficiente

**Como este script responde à pergunta:**
> Um negócio que cresce junto com a economia pode estar apenas surfando uma maré favorável — não construindo vantagem competitiva própria. Este script faz o teste de independência macroeconômica em duas frentes:
>
> 1. **Receita × IPCA:** Verifica se os períodos de maior receita coincidem com os de maior inflação. Uma correlação positiva forte pode indicar que o crescimento nominal é ilusório — a empresa vende mais em reais porque os preços subiram, não porque vendeu mais em volume.
> 2. **Receita × Desocupação PNADC:** Verifica se a receita cai quando o desemprego sobe. Uma correlação negativa forte indica que o negócio depende da renda disponível do consumidor — em recessões, seria duramente afetado.
>
> Para cada indicador, o script calcula a correlação de Pearson e o p-valor. Se p < 0,05 e a correlação for acima de ±0,5, o crescimento tem dependência macro relevante e o comprador precisa considerar cenários de reversão do ambiente externo.
>
> **Nota:** esta análise só executa se os arquivos de dados externos estiverem disponíveis em `data/externos/`. Caso contrário, uma mensagem de aviso é exibida e as demais análises continuam normalmente.

**Análise do Resultado:**
Esta análise separa o mérito da gestão do "vento a favor". Um negócio que cresce apenas porque o mercado inteiro cresceu (ex: o boom do e-commerce na pandemia) pode sofrer quando o cenário mudar. Queremos ver se a empresa-alvo ganha mercado (market share) independente das condições externas.


In [ ]:
macros = [
    ("ipca_pct",        "IPCA (% a.m.)",        COR_ALERTA),
    ("desocupacao_pct", "Desocupação PNADC (%)", COR_ROXO),
]
macros_presentes = [(col, lab, cor) for col, lab, cor in macros
                    if col in receita_mensal.columns and receita_mensal[col].notna().sum() > 5]

if not macros_presentes:
    print("[AVISO] Dados macro não disponíveis para esta análise.")
    print("Para ativar: certifique-se de que ipca_mensal.csv e desocupacao_pnadc.csv")
    print("estão em data/externos/ e foram gerados pelo script de conversão.")
else:
    n = len(macros_presentes)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 5))
    if n == 1: axes = [axes]
    fig.suptitle("Análise 8 — Receita × Indicadores Macroeconômicos", fontsize=13, fontweight="bold")

    for ax, (col_macro, lab_macro, cor_macro) in zip(axes, macros_presentes):
        df_plot = receita_mensal[["ano_mes", "receita_bruta", col_macro]].dropna()
        r_val, p_val = stats.pearsonr(df_plot["receita_bruta"], df_plot[col_macro])

        x_pos = range(len(df_plot))
        ax.bar(x_pos, df_plot["receita_bruta"] / 1000, color=COR_RECEITA, alpha=0.5, label="Receita")
        ax2_twin = ax.twinx()
        ax2_twin.plot(x_pos, df_plot[col_macro], color=cor_macro, linewidth=2,
                      marker="o", markersize=3, label=lab_macro)
        ax2_twin.set_ylabel(lab_macro, color=cor_macro)

        xtick = list(range(0, len(df_plot), 3))
        ax.set_xticks(xtick)
        ax.set_xticklabels([df_plot["ano_mes"].iloc[i] for i in xtick], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Receita (R$ mil)")
        sig = "(significativo)" if p_val < 0.05 else "(não significativo)"
        interp = "positiva — receita sobe com indicador" if r_val > 0.3 and p_val < 0.05 else \
                 "negativa — receita cai com indicador" if r_val < -0.3 and p_val < 0.05 else "fraca — independente da macro"
        ax.set_title(f"Receita × {lab_macro}\ncorr={r_val:.2f} {sig} — correlação {interp}", fontsize=10)

    plt.tight_layout()
    salvar(fig, "08_receita_vs_macro")
    plt.show()

    print("\n" + "="*55)
    print("INSIGHT — CORRELAÇÃO COM MACRO")
    print("="*55)
    for col_macro, lab_macro, _ in macros_presentes:
        df_c = receita_mensal[["receita_bruta", col_macro]].dropna()
        r_v, p_v = stats.pearsonr(df_c["receita_bruta"], df_c[col_macro])
        risco = "RISCO — crescimento macro-dependente" if abs(r_v) > 0.5 and p_v < 0.05 else "OK — crescimento estrutural"
        print(f"  {lab_macro:<30} corr={r_v:.2f} (p={p_v:.3f}) → {risco}")

---

## Análise 9 — A sazonalidade representa um padrão estrutural ou um risco de gestão de demanda?

> *"Identificar o índice de sazonalidade é fundamental para o planejamento de estoque, caixa e logística. O risco reside na incapacidade do sistema em absorver picos de demanda (como Black Friday) sem comprometer o nível de serviço (SLA). Quando os picos de receita coincidem com queda na satisfação e atrasos na entrega, o negócio gera faturamento no curto prazo ao custo de deteriorar a experiência do cliente — risco que o comprador herdaria diretamente."*

**Framework:** PDCA — análise de padrão e controle de demanda  
**Entrega:** Índice de sazonalidade mensal com amplitude pico/vale

**Como este script responde à pergunta:**
> Sazonalidade previsível é gerenciável. Sazonalidade descontrolada é risco operacional. Este script separa os dois cenários com duas visualizações:
>
> 1. **Índice de sazonalidade por mês:** Calcula a receita média de cada mês ao longo de todos os anos disponíveis e divide pela média geral do período. Um índice de 1,3 em novembro significa que novembro historicamente fatura 30% acima da média — útil para planejamento de estoque e logística. Barras verdes indicam meses de pico; vermelhas indicam vales.
> 2. **Pico de receita × degradação de SLA:** O scatter combina o índice de sazonalidade com o % de entregas no prazo em cada mês. O tamanho de cada bolha representa o volume de pedidos e a cor indica a nota média de review. Se os meses de pico (índice alto) aparecerem com % de prazo baixo e nota baixa, significa que a operação não escala bem — ela cresce em receita mas deteriora em qualidade, o que corrói a base de clientes a longo prazo.

**Análise do Resultado:** 
Sazonalidade previsível é gerenciável — sazonalidade não controlada é risco operacional. Em um modelo de e-commerce, o perigo não está no custo de estrutura ociosa, mas na incapacidade de absorver picos de demanda sem degradar o SLA e a experiência do cliente. Se os meses de maior receita coincidem com queda de prazo de entrega e satisfação, o negócio cresce em faturamento e perde em reputação simultaneamente — um ciclo que corrói a base de clientes no médio prazo e só se torna visível após a aquisição.

In [ ]:
MESES_ORD = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
             "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]

saz = (
    fato_entregues.groupby(["ano", "mes", "nome_mes"])
    .agg(receita=("preco", "sum"), n_pedidos=("id_pedido", "nunique"),
         nota_media=("nota_review", "mean"), pct_prazo=("entregue_no_prazo", "mean"))
    .reset_index()
)

indice_saz = (
    saz.groupby(["mes", "nome_mes"])
    .agg(receita_media=("receita", "mean"), pedidos_medio=("n_pedidos", "mean"),
         nota_media=("nota_media", "mean"), prazo_medio=("pct_prazo", "mean"))
    .reset_index().sort_values("mes")
)
indice_saz["indice_saz"]  = indice_saz["receita_media"] / indice_saz["receita_media"].mean()
indice_saz["prazo_medio"] = indice_saz["prazo_medio"] * 100
indice_saz["nome_mes_ord"] = pd.Categorical(indice_saz["nome_mes"], categories=MESES_ORD, ordered=True)
indice_saz = indice_saz.sort_values("nome_mes_ord")

meses_abrev = [m[:3] for m in indice_saz["nome_mes"].tolist()]
x_m = range(len(indice_saz))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Análise 9 — Sazonalidade: Padrão Estrutural ou Risco de Demanda?", fontsize=13, fontweight="bold")

# Índice de sazonalidade
cores_saz = [COR_MARGEM if v >= 1.0 else COR_ALERTA for v in indice_saz["indice_saz"]]
axes[0].bar(x_m, indice_saz["indice_saz"], color=cores_saz, alpha=0.85)
axes[0].axhline(1.0, color="black", linewidth=1)
for xi, val in zip(x_m, indice_saz["indice_saz"]):
    axes[0].text(xi, val + 0.01, f"{val:.2f}x", ha="center", va="bottom", fontsize=8)
axes[0].set_xticks(x_m); axes[0].set_xticklabels(meses_abrev, fontsize=9)
axes[0].set_ylabel("Índice (1.0 = média do período)")
axes[0].set_title("Índice de Sazonalidade de Receita", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_MARGEM, label="Acima da média (pico)"),
    mpatches.Patch(color=COR_ALERTA, label="Abaixo da média (vale)"),
], frameon=False, fontsize=8)

# Scatter: pico de receita × SLA
sc = axes[1].scatter(
    indice_saz["indice_saz"], indice_saz["prazo_medio"],
    s=indice_saz["pedidos_medio"] / indice_saz["pedidos_medio"].max() * 400 + 30,
    c=indice_saz["nota_media"], cmap="RdYlGn", vmin=3, vmax=5,
    alpha=0.85, edgecolors="white", linewidth=0.5, zorder=5
)
plt.colorbar(sc, ax=axes[1], label="Nota média de review")
for _, row in indice_saz.iterrows():
    axes[1].annotate(row["nome_mes"][:3], (row["indice_saz"], row["prazo_medio"]),
                     fontsize=8, ha="center", xytext=(0, 5), textcoords="offset points")
axes[1].axhline(90, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5, label="SLA 90%")
axes[1].axvline(1.0, color=COR_NEUTRO, linestyle=":",  linewidth=1, alpha=0.5)
axes[1].set_xlabel("Índice de Sazonalidade")
axes[1].set_ylabel("% Entregue no Prazo")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Picos de Receita × Degradação de SLA\n(tamanho = volume | cor = satisfação)", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "09_sazonalidade")
plt.show()

pico  = indice_saz.loc[indice_saz["indice_saz"].idxmax()]
vale  = indice_saz.loc[indice_saz["indice_saz"].idxmin()]
ampl  = pico["indice_saz"] / vale["indice_saz"]
meses_pico_sla = indice_saz[(indice_saz["indice_saz"] > 1.1) & (indice_saz["prazo_medio"] < 85)]

print("\n" + "="*55)
print("INSIGHT — SAZONALIDADE")
print("="*55)
print(f"Mês de pico     : {pico['nome_mes']} ({pico['indice_saz']:.2f}x a média)")
print(f"Mês de vale     : {vale['nome_mes']} ({vale['indice_saz']:.2f}x a média)")
print(f"Amplitude       : {ampl:.1f}x — {'alta' if ampl > 3 else 'moderada' if ampl > 2 else 'baixa'}")
if len(meses_pico_sla):
    print(f"\nMeses com pico de receita E SLA degradado:")
    for _, r in meses_pico_sla.iterrows():
        print(f"  {r['nome_mes']}: índice {r['indice_saz']:.2f}x | SLA: {r['prazo_medio']:.1f}%")
else:
    print("\nSem meses com pico de receita e SLA degradado simultaneamente.")

---

## Análise 10 — Quantos clientes respondem por 80% da receita — há concentração de risco na base de compradores?

> *"A utilização da Curva de Lorenz e do Coeficiente de Gini quantifica o nível de dependência que o negócio tem de uma parcela reduzida da base de clientes. Uma base altamente concentrada é frágil; a perda de um pequeno grupo de compradores "âncoras" pode inviabilizar a operação. A diversificação da carteira é um dos principais indicadores de resiliência e estabilidade do faturamento futuro que um investidor deve validar"*

**Framework:** Curva de Lorenz + Princípio de Pareto aplicado à base de clientes  
**Entrega:** Curva de Lorenz com coeficiente de Gini + quadrante frequência × valor

**Como este script responde à pergunta:**
> Para medir concentração da base de clientes, o script calcula a contribuição individual de cada cliente e constrói duas visualizações complementares:
>
> 1. **Curva de Lorenz:** Ordena os clientes do maior para o menor em receita acumulada. Quanto mais a curva se afasta da diagonal de igualdade, maior a concentração. O coeficiente de Gini (0 = igualdade total, 1 = concentração máxima) quantifica a desigualdade em um único número. O ponto onde a curva cruza 80% indica exatamente quantos clientes sustentam o negócio.
> 2. **Quadrante frequência × valor:** Classifica cada cliente em quatro perfis — Âncora (alto valor + frequente), Ocasional de alto ticket, Regular de baixo ticket e Marginal. A concentração no quadrante de alto valor + baixa frequência é o sinal mais preocupante: esses clientes geram muita receita mas podem não voltar.

**Análise do Resultado:**
 Baseado no princípio de Pareto. Se poucos clientes sustentam o faturamento, a perda de um único contrato ou a mudança de hábito de um pequeno grupo pode falir a empresa. Um negócio saudável deve ter uma base diversificada de compradores para diluir o risco.


In [ ]:
# ─── Análise 10 — Concentração de Receita na Base de Clientes ────────────────
import numpy as np

cliente_receita = (
    fato_entregues
    .groupby("id_cliente")
    .agg(
        receita_total = ("preco",     "sum"),
        n_pedidos     = ("id_pedido", "nunique"),
    )
    .reset_index()
    .sort_values("receita_total", ascending=False)
)
total_clientes = len(cliente_receita)
total_receita  = cliente_receita["receita_total"].sum()
cliente_receita["receita_acum_pct"]  = cliente_receita["receita_total"].cumsum() / total_receita * 100
cliente_receita["clientes_acum_pct"] = np.arange(1, total_clientes + 1) / total_clientes * 100

n_cli_80pct   = int((cliente_receita["receita_acum_pct"] <= 80).sum()) + 1
pct_cli_80pct = n_cli_80pct / total_clientes * 100

# Gini
receita_ord = np.sort(cliente_receita["receita_total"].values)
n = len(receita_ord)
gini = (2 * np.sum(np.arange(1, n+1) * receita_ord) / (n * receita_ord.sum())) - (n + 1) / n

# Perfis
med_freq = cliente_receita["n_pedidos"].median()
med_val  = cliente_receita["receita_total"].median()

def perfil_cliente(row):
    af = row["n_pedidos"]     >= med_freq
    av = row["receita_total"] >= med_val
    if af and av:     return "Âncora (alto valor + frequente)"
    if not af and av: return "Ocasional de alto ticket"
    if af and not av: return "Regular de baixo ticket"
    return "Marginal"

cliente_receita["perfil"] = cliente_receita.apply(perfil_cliente, axis=1)

perfis_ordem = [
    "Âncora (alto valor + frequente)",
    "Ocasional de alto ticket",
    "Regular de baixo ticket",
    "Marginal",
]
perfis_cores = {
    "Âncora (alto valor + frequente)": COR_RECEITA,
    "Ocasional de alto ticket":         COR_MARGEM,
    "Regular de baixo ticket":          COR_NEUTRO,
    "Marginal":                         "#cccccc",
}
contagem_perfis = cliente_receita["perfil"].value_counts()
receita_perfis  = cliente_receita.groupby("perfil")["receita_total"].sum() / total_receita * 100

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("Análise 10 — Composição da Base de Clientes por Perfil de Valor", fontsize=13, fontweight="bold")

# Painel 1: Curva de Lorenz
axes[0].plot(cliente_receita["clientes_acum_pct"], cliente_receita["receita_acum_pct"],
             color=COR_RECEITA, linewidth=2, label="Curva real")
axes[0].plot([0, 100], [0, 100], color=COR_NEUTRO, linestyle="--", linewidth=1, label="Igualdade perfeita")
axes[0].axhline(80, color=COR_ALERTA, linestyle=":", linewidth=1, alpha=0.7)
axes[0].axvline(pct_cli_80pct, color=COR_ALERTA, linestyle=":", linewidth=1, alpha=0.7)
axes[0].annotate(
    f"{pct_cli_80pct:.1f}% dos clientes\ngeram 80% da receita",
    xy=(pct_cli_80pct, 80), xytext=(pct_cli_80pct + 8, 62),
    arrowprops=dict(arrowstyle="->", color="black", lw=1),
    fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=COR_ALERTA, alpha=0.9)
)
axes[0].fill_between(cliente_receita["clientes_acum_pct"],
                     cliente_receita["receita_acum_pct"],
                     cliente_receita["clientes_acum_pct"],
                     alpha=0.08, color=COR_RECEITA)
axes[0].set_xlabel("% acumulado de clientes")
axes[0].set_ylabel("% acumulado de receita")
axes[0].set_title(f"Curva de Lorenz\n(Gini = {gini:.2f})", fontsize=11)
axes[0].legend(frameon=False, fontsize=9)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))

# Painel 2: Clientes por perfil
n_vals   = [contagem_perfis.get(p, 0) for p in perfis_ordem]
cores_b  = [perfis_cores[p] for p in perfis_ordem]
labels_b = ["Âncora", "Ocasional\nalto ticket", "Regular\nbaixo ticket", "Marginal"]

bars2 = axes[1].bar(labels_b, n_vals, color=cores_b, alpha=0.85, width=0.55)
for bar, v in zip(bars2, n_vals):
    if v > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(n_vals)*0.01,
                     f"{v:,}", ha="center", va="bottom", fontsize=9, fontweight="500")
axes[1].set_ylabel("Nº de clientes")
axes[1].set_title("Clientes por perfil", fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v/1000)}K" if v >= 1000 else str(int(v))))
axes[1].set_ylim(0, max(n_vals) * 1.15)
for spine in ["top", "right"]:
    axes[1].spines[spine].set_visible(False)

# Painel 3: Receita por perfil
r_vals = [receita_perfis.get(p, 0) for p in perfis_ordem]
bars3  = axes[2].bar(labels_b, r_vals, color=cores_b, alpha=0.85, width=0.55)
for bar, v in zip(bars3, r_vals):
    if v > 0.5:
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="500")
axes[2].set_ylabel("% da receita total")
axes[2].set_title("Participação na receita por perfil", fontsize=11)
axes[2].set_ylim(0, 100)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
for spine in ["top", "right"]:
    axes[2].spines[spine].set_visible(False)

plt.tight_layout()
salvar(fig, "10_concentracao_clientes")
plt.show()

# Ticket médio por perfil
ticket_perfil = cliente_receita.groupby("perfil")["receita_total"].mean()

print("\n" + "="*55)
print("INSIGHT — COMPOSIÇÃO DA BASE DE CLIENTES")
print("="*55)
print(f"Total de clientes únicos : {total_clientes:,}")
print(f"Clientes p/ 80% receita  : {n_cli_80pct:,} ({pct_cli_80pct:.1f}% da base)")
print(f"Coeficiente de Gini      : {gini:.2f}")
print()
print(f"{'Perfil':<38} {'Clientes':>8} {'% Receita':>10} {'Ticket médio':>14}")
print("-"*74)
for p in perfis_ordem:
    n_p = contagem_perfis.get(p, 0)
    r_p = receita_perfis.get(p, 0)
    t_p = ticket_perfil.get(p, 0)
    print(f"{p:<38} {n_p:>8,} {r_p:>9.1f}% {t_p:>13,.2f}")


---

## Análise 11 — A base de clientes cresce por retenção ou por aquisição constante — há churn silencioso?

> *"Através da Análise de Coorte (Cohort), é possível distinguir o crescimento real de um modelo baseado em "balde furado", onde a entrada de novos clientes apenas mascara a perda dos antigos. A retenção é o indicador mais preciso de qualidade do produto e da experiência do cliente. Um negócio que depende exclusivamente de aquisição constante possui baixa previsibilidade de receita futura, enquanto retenção alta indica que a base existente continua gerando valor — ativo intangível relevante para o comprador avaliar antes de fechar o negócio."*

**Framework:** Cohort Retention Analysis (Análise de Coorte de Retenção)  
**Entrega:** Heatmap de retenção por coorte mensal + curva de recompra acumulada

**Como este script responde à pergunta:**
> A análise de coorte é a ferramenta mais precisa para distinguir um negócio saudável de um negócio com churn silencioso. Este script constrói duas visualizações:
>
> 1. **Heatmap de retenção por coorte:** Cada linha é um grupo de clientes que fizeram sua primeira compra no mesmo mês. As colunas mostram quantos desses clientes voltaram nos meses seguintes. Se as células após M+1 forem próximas de zero, o negócio tem estrutura de compra única. Calor persistente após M+3 indica retenção real.
> 2. **Curva de recompra acumulada:** Mostra em quantos dias uma fração da base faz a segunda compra. O ponto onde a curva achata representa o teto de recompra — se achata em percentuais baixos, o negócio depende inteiramente de aquisição constante de novos clientes, modelo de alto custo e baixa previsibilidade.

**Análise do Resultado:** 
É muito mais barato manter um cliente do que conquistar um novo. Se a empresa precisa gastar fortunas em marketing o tempo todo porque os clientes compram uma vez e nunca mais voltam, o modelo é um "balde furado". A retenção alta prova que o produto tem qualidade e gera fidelidade, o que é o maior ativo para um investidor.

In [ ]:
# ─── Análise 11 — Churn Silencioso: Retenção vs Aquisição ────────────────────
import warnings
warnings.filterwarnings("ignore")

compras = fato_entregues[["id_cliente", "data_compra"]].copy()
compras["ano_mes_compra"] = compras["data_compra"].dt.to_period("M")
primeira_compra = compras.groupby("id_cliente")["ano_mes_compra"].min().rename("coorte")
compras = compras.join(primeira_compra, on="id_cliente")
compras["mes_desde_coorte"] = (compras["ano_mes_compra"] - compras["coorte"]).apply(lambda x: x.n)

# Curva de recompra acumulada
cli_multi = (
    compras.groupby("id_cliente")["data_compra"]
    .apply(lambda x: sorted(x.tolist())).reset_index()
)
cli_multi["n_compras"] = cli_multi["data_compra"].apply(len)
total_cli = len(cli_multi)
segundas  = cli_multi[cli_multi["n_compras"] >= 2].copy()
segundas["dias_ate_recompra"] = segundas["data_compra"].apply(lambda x: (x[1] - x[0]).days)
pct_recompra = len(segundas) / total_cli * 100

dias_sort = np.sort(segundas["dias_ate_recompra"].values)
curva_acum = np.arange(1, len(dias_sort) + 1) / total_cli * 100

pct_30  = float(curva_acum[dias_sort <= 30].max())  if any(dias_sort <= 30)  else 0
pct_90  = float(curva_acum[dias_sort <= 90].max())  if any(dias_sort <= 90)  else 0
pct_180 = float(curva_acum[dias_sort <= 180].max()) if any(dias_sort <= 180) else 0

# Retenção por coorte (para métricas do insight)
coorte_size = compras.groupby("coorte")["id_cliente"].nunique().rename("n_clientes")
retencao_raw = (
    compras.groupby(["coorte", "mes_desde_coorte"])["id_cliente"]
    .nunique().reset_index(name="retidos")
)
retencao_raw = retencao_raw.join(coorte_size, on="coorte")
retencao_raw["pct_retido"] = retencao_raw["retidos"] / retencao_raw["n_clientes"] * 100
coortes_validas = coorte_size[coorte_size >= 30].index
ret_pivot = (
    retencao_raw[retencao_raw["mes_desde_coorte"] <= 12]
    .pivot(index="coorte", columns="mes_desde_coorte", values="pct_retido")
)
ret_pivot = ret_pivot.loc[ret_pivot.index.isin(coortes_validas)]
ret_m1 = ret_pivot[1].mean() if 1 in ret_pivot.columns else 0
ret_m3 = ret_pivot[3].mean() if 3 in ret_pivot.columns else 0

# ─── Funil ────────────────────────────────────────────────────────────────────
n_voltaram     = len(segundas)
n_voltou_30d   = int((segundas["dias_ate_recompra"] <= 30).sum())
n_retido_m1    = int(ret_pivot[1].mean() / 100 * total_cli) if 1 in ret_pivot.columns else 0
pct_voltou_30d = n_voltou_30d / total_cli * 100

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 11 — Churn Silencioso: Retenção vs Aquisição Constante", fontsize=13, fontweight="bold")

# Painel 1: Funil
etapas = [
    (f"100 clientes novos",         100,                   COR_RECEITA,   "100%"),
    (f"Voltaram alguma vez",         round(pct_recompra),  COR_ROXO,      f"{pct_recompra:.1f}%"),
    (f"Voltaram em até 30 dias",     round(pct_30),       COR_MARGEM,    f"{pct_30:.1f}%"),
    (f"Retidos no M+1 (mês seguinte)", max(round(ret_m1), 0), COR_NEUTRO, f"{ret_m1:.1f}%"),
]
y_pos = list(range(len(etapas)))
for idx_e, (label, val, cor, pct_label) in enumerate(etapas):
    largura = max(val / 100, 0.02)
    axes[0].barh(idx_e, largura, color=cor, alpha=0.85, height=0.6)
    axes[0].text(largura + 0.01, idx_e, pct_label, va="center", fontsize=10, fontweight="500")
    axes[0].text(-0.01, idx_e, label, va="center", ha="right", fontsize=9,
                 color="gray" if idx_e > 0 else "black")

axes[0].set_xlim(-0.01, 1.25)
axes[0].set_ylim(-0.6, len(etapas) - 0.4)
axes[0].invert_yaxis()
axes[0].axis("off")
axes[0].set_title("O que acontece com cada 100 clientes novos", fontsize=11)

# Painel 2: Curva de recompra acumulada
axes[1].plot(dias_sort, curva_acum, color=COR_RECEITA, linewidth=2, label="Recompra acumulada real")
axes[1].fill_between(dias_sort, curva_acum, alpha=0.12, color=COR_RECEITA)

# Linha de referência saudável (30%)
axes[1].axhline(30, color=COR_ALERTA, linestyle="--", linewidth=1.5, alpha=0.8,
                label="Referência saudável (30%+)")

# Marcos de tempo
for marco, cor_m, lbl in [(30, COR_DESTAQUE, f"30d\n{pct_30:.1f}%"),
                           (90, COR_ROXO,    f"90d\n{pct_90:.1f}%"),
                           (180, COR_NEUTRO, f"180d\n{pct_180:.1f}%")]:
    pct_m = float(curva_acum[dias_sort <= marco].max()) if any(dias_sort <= marco) else 0
    axes[1].axvline(marco, color=cor_m, linestyle=":", linewidth=1.2, alpha=0.8)
    axes[1].annotate(lbl, xy=(marco, pct_m), xytext=(marco + 5, pct_m + 2),
                     fontsize=8, color=cor_m)

axes[1].set_xlabel("Dias desde a primeira compra")
axes[1].set_ylabel("% acumulado da base que recomprou")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Curva de recompra acumulada\n(% da base que fez 2ª compra em até X dias)", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)
axes[1].set_xlim(0, min(365, dias_sort.max()))

plt.tight_layout()
salvar(fig, "11_retencao_churn")
plt.show()

sinal_ret = "recompra recorrente" if pct_recompra > 30 else "modelo de compra única — churn silencioso"

print("\n" + "="*55)
print("INSIGHT — RETENÇÃO E CHURN")
print("="*55)
print(f"Total clientes únicos    : {total_cli:,}")
print(f"Clientes com recompra    : {len(segundas):,} ({pct_recompra:.1f}% da base)")
print(f"Recompra em até 30 dias  : {pct_30:.1f}%")
print(f"Recompra em até 90 dias  : {pct_90:.1f}%")
print(f"Retenção média M+1       : {ret_m1:.1f}%")
print(f"Retenção média M+3       : {ret_m3:.1f}%")
print(f"Sinal de retenção        : {sinal_ret}")


---

## Síntese do Bloco 1 — Viabilidade Econômica

Esta seção consolida os principais achados das nove análises realizadas neste bloco e apresenta o veredicto parcial para orientar a continuidade da investigação.
> **Limitações desta análise:** ausência de custo operacional direto limita a análise de margem real — o frete é utilizado como proxy de pressão de custo variável. Crescimento de receita não implica necessariamente lucratividade. A correlação estatística indica associação, não causalidade.


In [ ]:
# ─── Cálculos para o veredicto ───────────────────────────────────────────────
# Todos os valores são recalculados aqui para garantir que esta célula
# funcione independentemente da ordem de execução do notebook.

rm_valido        = receita_mensal.dropna(subset=["crescimento_mom"])
meses_positivos  = (rm_valido["crescimento_mom"] > 0).sum()
meses_total      = len(rm_valido)
pct_meses_pos    = meses_positivos / meses_total

receita_primeiro = receita_mensal["receita_bruta"].iloc[0]
receita_ultimo   = receita_mensal["receita_bruta"].iloc[-1]
mes_primeiro     = receita_mensal["ano_mes"].iloc[0]
mes_ultimo       = receita_mensal["ano_mes"].iloc[-1]

frete_geral_pct    = fato_entregues["valor_frete"].sum() / fato_entregues["preco"].sum() * 100
frete_primeiro_tri = margem_trimestral.iloc[0]["pct_frete_receita"]
frete_ultimo_tri   = margem_trimestral.iloc[-1]["pct_frete_receita"]
frete_pico_tri     = margem_trimestral.iloc[-1]["label"]
frete_trend_up     = frete_ultimo_tri > frete_primeiro_tri
frete_delta_pp     = frete_ultimo_tri - frete_primeiro_tri

ticket_coef      = np.polyfit(list(range(len(receita_mensal))), receita_mensal["ticket_medio"].fillna(method="ffill"), 1)[0]
ticket_trend_up  = ticket_coef > 0
ticket_primeiro  = receita_mensal["ticket_medio"].dropna().iloc[0]
ticket_ultimo    = receita_mensal["ticket_medio"].dropna().iloc[-1]
ticket_delta_pct = (ticket_ultimo - ticket_primeiro) / ticket_primeiro * 100

# Concentração — recalcula localmente para não depender da Análise 6
_margem_cat = (
    fato_entregues
    .groupby("nome_categoria_produto")
    .agg(receita_bruta=("preco", "sum"))
    .reset_index()
    .sort_values("receita_bruta", ascending=False)
    .reset_index(drop=True)
)
_margem_cat["pct_receita"] = _margem_cat["receita_bruta"] / _margem_cat["receita_bruta"].sum() * 100
_margem_cat["pct_acum"]    = _margem_cat["pct_receita"].cumsum()
n_cats_80   = (_margem_cat["pct_acum"] <= 80).sum() + 1
n_cats_total = len(_margem_cat)
top3_cat_pct = _margem_cat.head(3)["pct_receita"].sum()

_margem_estado = (
    fato_entregues
    .groupby("estado_cliente")
    .agg(receita_bruta=("preco", "sum"))
    .reset_index()
    .sort_values("receita_bruta", ascending=False)
    .reset_index(drop=True)
)
_margem_estado["pct_receita"] = _margem_estado["receita_bruta"] / _margem_estado["receita_bruta"].sum() * 100
_margem_estado["pct_acum"]    = _margem_estado["pct_receita"].cumsum()
sp_rj_mg_pct = _margem_estado[_margem_estado["estado_cliente"].isin(["SP","RJ","MG"])]["pct_receita"].sum()
n_estados_80 = (_margem_estado["pct_acum"] <= 80).sum() + 1
n_estados_total = len(_margem_estado)


# Concentração de clientes (Análise 10) — recalcula localmente
_cli_rec = (
    fato_entregues.groupby("id_cliente")["preco"].sum()
    .sort_values(ascending=False).reset_index(name="receita")
)
_cli_rec["pct_acum"] = _cli_rec["receita"].cumsum() / _cli_rec["receita"].sum() * 100
_n_cli_80 = int((_cli_rec["pct_acum"] <= 80).sum()) + 1
_pct_cli_80 = _n_cli_80 / len(_cli_rec) * 100

# Recompra (Análise 11) — recalcula localmente
_compras_v = fato_entregues[["id_cliente","data_compra"]].copy()
_compras_v["ym"] = _compras_v["data_compra"].dt.to_period("M")
_n_recompra = _compras_v.groupby("id_cliente")["ym"].nunique()
_pct_recompra_v = (_n_recompra >= 2).sum() / len(_n_recompra) * 100

# ─── Semáforos ────────────────────────────────────────────────────────────────
p1 = "✅ SIM"        if pct_meses_pos > 0.6   else "⚠️  PARCIAL"
p2 = "⚠️  PARCIAL"  if frete_trend_up         else "✅ SIM"
p3 = "⚠️  ATENÇÃO"  if frete_trend_up         else "✅ OK"
p4 = "✅ SIM"        if pct_meses_pos > 0.6   else "⚠️  PARCIAL"
p5 = "⚠️  LIMÍTROFE" if sp_rj_mg_pct > 60    else "✅ OK"
p6 = "✅ SIM"        if ticket_trend_up        else "⚠️  QUEDA"
p7 = "⚠️  CONCENTRADO" if _pct_cli_80 < 10        else "✅ DIVERSIFICADO"
p8 = "⚠️  COMPRA ÚNICA" if _pct_recompra_v < 20   else "✅ HÁ RETENÇÃO"

# ─── Textos 100% derivados de variáveis ──────────────────────────────────────
cresc_padrão  = "consistente" if pct_meses_pos > 0.6 else "irregular"
frete_dir     = "crescente"   if frete_trend_up       else "estável"
frete_impacto = "pode estar comprimindo a conversão"  if frete_trend_up else "não apresenta sinal de barreira ativa"
frete_alerta  = "sinal de alerta — monitorar evolução" if frete_trend_up else "sem deterioração identificada"
geo_avaliação = "limítrofe — risco de dependência regional relevante" if sp_rj_mg_pct > 60 else "dentro do aceitável"
ticket_dir    = "positiva"    if ticket_trend_up       else "de queda"
ticket_interp = "crescimento agrega valor por transação" if ticket_trend_up else "crescimento pode estar diluindo valor por transação"

# ─── Sinal geral ──────────────────────────────────────────────────────────────
n_alertas = sum([frete_trend_up, pct_meses_pos <= 0.6, sp_rj_mg_pct > 60, not ticket_trend_up, _pct_cli_80 < 10, _pct_recompra_v < 20])
if n_alertas == 0:
    sinal = "✅ POSITIVO — fundamentos sólidos"
elif n_alertas <= 2:
    sinal = "⚠️  POSITIVO COM CONDICIONANTE"
else:
    sinal = "🔴 ATENÇÃO — múltiplos sinais de risco"

# ─── Impressão ────────────────────────────────────────────────────────────────
print("=" * 65)
print("SÍNTESE — BLOCO 1: VIABILIDADE ECONÔMICA")
print("=" * 65)

print(f"""
[ CRESCIMENTO DE RECEITA ]
  Período analisado       : {mes_primeiro} → {mes_ultimo}
  Receita total           : R$ {fato_entregues["preco"].sum():>14,.0f}
  Receita no 1º mês       : R$ {receita_primeiro:>14,.0f}
  Receita no último mês   : R$ {receita_ultimo:>14,.0f}
  Múltiplo de crescimento : {receita_ultimo / receita_primeiro:.1f}x
  Ticket médio global     : R$ {fato_entregues["preco"].mean():>14,.2f}
  Meses de crescimento    : {meses_positivos} de {meses_total} ({pct_meses_pos*100:.0f}%)

[ MARGEM E FRETE ]
  % Frete / Receita geral : {frete_geral_pct:.1f}%
  % Frete no 1º trimestre : {frete_primeiro_tri:.1f}%
  % Frete no último trim. : {frete_ultimo_tri:.1f}% ({frete_pico_tri})
  Variação do período     : {frete_delta_pp:+.1f}pp — tendência {frete_dir}

[ TICKET MÉDIO ]
  Ticket no início        : R$ {ticket_primeiro:,.2f}
  Ticket no final         : R$ {ticket_ultimo:,.2f}
  Variação                : {ticket_delta_pct:+.1f}% — tendência {ticket_dir}

[ CONCENTRAÇÃO ]
  SP + RJ + MG            : {sp_rj_mg_pct:.1f}% da receita
  Top 3 categorias        : {top3_cat_pct:.1f}% da receita
  Categorias para 80%     : {n_cats_80} de {n_cats_total}
  Estados para 80%        : {n_estados_80} de {n_estados_total}

[ VEREDICTO DAS PERGUNTAS ]
  {p1} Crescimento consistente com tendência clara?
       Receita passou de R$ {receita_primeiro:,.0f} para R$ {receita_ultimo:,.0f}
       ({receita_ultimo/receita_primeiro:.1f}x) em {meses_total} meses.
       Padrão {cresc_padrão}: {meses_positivos} de {meses_total} meses positivos ({pct_meses_pos*100:.0f}%).

  {p2} Margem operacional se manteve com o crescimento?
       % frete saiu de {frete_primeiro_tri:.1f}% para {frete_ultimo_tri:.1f}% ({frete_delta_pp:+.1f}pp).
       {frete_alerta}.

  {p3} Frete representa barreira de demanda?
       Frete {frete_dir} — {frete_impacto}.
       Investigação aprofundada na Fase 2 — Logística.

  {p4} Há evidências de tração sustentável?
       {meses_positivos} de {meses_total} meses com crescimento positivo — padrão {cresc_padrão}.

  {p5} Concentração geográfica acima de 60%?
       SP+RJ+MG = {sp_rj_mg_pct:.1f}% — {geo_avaliação}.

  {p6} Ticket médio crescendo com o volume?
       Ticket foi de R$ {ticket_primeiro:,.2f} para R$ {ticket_ultimo:,.2f}
       ({ticket_delta_pct:+.1f}%) — tendência {ticket_dir}: {ticket_interp}.

  {p7} Concentração na base de clientes controlada?
       {_n_cli_80:,} clientes ({_pct_cli_80:.1f}% da base) geram 80% da receita.

  {p8} Base cresce por retenção (não apenas aquisição)?
       {_pct_recompra_v:.1f}% dos clientes realizaram ao menos 2 compras.
""")

print("=" * 65)
print("VEREDICTO PARCIAL DO BLOCO 1")
print("=" * 65)
print(f"""
Sinal geral : {sinal}
Próximo passo → Bloco 2: qualidade do motor de receita.
""")


---
*Próximo notebook: `02_EDA_motor_receita.ipynb` — O motor de receita é robusto ou perigosamente concentrado?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
